# Spatial Copilot — YOLO11-seg fine-tune on Colab

Runtime → **Change runtime type** → GPU (T4 is enough) before running.

Same steps as `COLAB.md`. The dataset is rebuilt from HuggingFace; nothing large is in git.

## 1. Clone the repo + install training deps

Do **not** `pip install -r requirements.txt` here — it pins CPU torch and would replace Colab's CUDA build.

In [ ]:
!git clone https://github.com/joshyajith863/Spatial_model.git
%cd Spatial_model
!pip install -q ultralytics pandas pyarrow huggingface_hub pyyaml
!nvidia-smi

In [ ]:
# Verify the GPU reaches torch - and repair a CPU-only wheel.
# Diagnosis first: no point downloading ~2.5 GB if the runtime has no GPU at all.
import shutil
import subprocess
import sys

import torch

print("torch          :", torch.__version__)
print("built for cuda :", torch.version.cuda)
print("cuda available :", torch.cuda.is_available())

smi = shutil.which("nvidia-smi")
if smi is None:
    print("\nNo nvidia-smi -> this runtime has NO GPU attached.")
    print("Runtime -> Change runtime type -> T4 GPU, then re-run this cell.")
elif torch.cuda.is_available():
    subprocess.run([smi, "--query-gpu=name,memory.total", "--format=csv"])
else:
    free = shutil.disk_usage("/").free / 1e9
    print(f"\nCPU-only torch wheel detected. Free disk: {free:.1f} GB (need ~6 GB).")
    print("Downloading + installing the CUDA build. This is 2-3 GB and takes 3-10 min,")
    print("so expect a long silent-looking wait - the progress bar below is real.\n")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision"], check=False)
    rc = subprocess.run([sys.executable, "-m", "pip", "install", "--force-reinstall",
                         "--no-cache-dir", "torch", "torchvision"], check=False).returncode
    print("\npip exit code:", rc)
    print("** Restart the runtime now (Runtime -> Restart session), then re-run this cell. **")
    print("Expect: cuda available : True")
    print("Still False after restart? Pin the wheel:")
    print("  pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126")


## 2. Build the CubiCasa5K YOLO dataset

~1.5 GB download, 5–10 min. Writes `data/yolo/data.yaml` with the local root.

In [ ]:
!python scripts/convert_cubicasa_yolo.py --split both

## 3. Smoke test (200 images, 2 epochs)

Confirms the dataset/model/GPU path before burning an hour.

In [ ]:
!python scripts/train_yolo.py --device 0 --epochs 2 --batch 16 --imgsz 640 --limit 200

---
## 4. Train into Drive (so a disconnect costs you one epoch, not the run)

Ultralytics already saves every epoch: `weights/last.pt` (exact resume point),
`weights/best.pt`, and `results.csv`. There is nothing to hand-roll — the only thing
that matters is *where* it writes. Point `--project` at Drive and the checkpoints are
already persistent.

`--save-period 5` additionally keeps `epoch5.pt`, `epoch10.pt`, ... so you can roll back
to an earlier point instead of only the latest.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

RUN_ROOT = Path('/content/drive/MyDrive/spatial_runs/segment')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("checkpoints will live in:", RUN_ROOT)
print("free space on Drive   :", end=" ")
!df -h /content/drive | tail -1


In [ ]:
# Loader variant: live per-epoch progress bars, resumed-safe output dir.
!python main.py train --device 0 --epochs 50 --batch 16 --save-period 5 \
    --project /content/drive/MyDrive/spatial_runs/segment --name cubicasa


### 4b. Resume an interrupted run


In [ ]:
# Resume after a disconnect / timeout. Safe to run when nothing was interrupted:
# it only fires when a checkpoint from an unfinished run exists.
from pathlib import Path

run_dir = Path('/content/drive/MyDrive/spatial_runs/segment/cubicasa')
ckpt = run_dir / 'weights' / 'last.pt'

if ckpt.exists():
    size_mb = ckpt.stat().st_size / 1e6
    print(f"found {ckpt} ({size_mb:.1f} MB) - continuing where it stopped")
    !python main.py train --resume "{ckpt}" --device 0
else:
    print("no checkpoint yet - run the training cell above first")
    print("expected at:", ckpt)


## 5. Evaluate per class (floor / wall / door / window)

In [ ]:
!python scripts/eval_yolo.py --weights /content/drive/MyDrive/spatial_runs/cubicasa/weights/best.pt

## 6. Download the weights to your laptop

Put `best.pt` in the repo's `weights/` folder, then locally:
`python main.py load yolo --weights weights/best.pt`

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/spatial_runs/cubicasa/weights/best.pt')